# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library, referencing all data structures via their explicit `@id` fields for traceability and reproducibility.

### Dataset Source
The dataset is described via a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and curated for FAIR clinical and biomarker research in oncology.

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. The dataset is referenced by its schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Discover the available record sets and their fields. All entities are referenced via their `@id`.

Note: Record set `@id`s are resolved from the dataset metadata. Fields of interest (columns/attributes) are referenced by their own `@id` for subsequent operations.

In [ ]:
# List all record sets with their @id and fields
record_sets = dataset.record_sets

print("Available record sets and fields:")
for rec in record_sets:
    print(f"\nRecord set name: {rec.name}")
    print(f"@id: {rec['@id']}")
    print("Fields:")
    for field in rec.fields:
        print(f"  - {field.name} (@id: {field['@id']})")

## 3. Data Extraction

For analysis, load data from each record set using its `@id`. All values for selection and processing will reference these identifiers.

First, we list the `@id`s. Then, we extract all data for each record set into a dictionary of pandas DataFrames.

In [ ]:
# Collect all record set @ids
record_sets = dataset.record_sets
record_set_ids = [rec['@id'] for rec in record_sets]
print("Record set @ids:")
for id_ in record_set_ids:
    print(f"  - {id_}")

# Load each record set into a pandas DataFrame
dataframes = {}
for rec in record_sets:
    rec_id = rec['@id']
    data = list(dataset.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(data)

# Show columns and a preview for the main record set
if record_set_ids:
    main_set_id = record_set_ids[0]
    print(f"\nFirst record set columns (@id: {main_set_id}):")
    print(dataframes[main_set_id].columns.tolist())
    display(dataframes[main_set_id].head())
else:
    print("No record sets available.")

## 4. Exploratory Data Analysis (EDA)

This section demonstrates filtering, normalization, and grouping using the field `@id`s as specified in the schema. Choose a numeric field and a group field based on the field overview above.

Below, we showcase:
- Filtering records for numeric values above a threshold
- Normalizing a field
- Grouping by a categorical attribute

Adjust field `@id`s below to those appropriate for your selected analysis targets.

In [ ]:
# Example: Use numeric field and group/categorical field @ids based on the record set
# Please inspect the output above to set the appropriate field @ids below.

# For demonstration, let us assume:
# - The main record set @id is main_set_id
# - Suppose the field '@id' for "Age at Second CRC Diagnosis" is 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/field_age_at_second_crc'
# - And a grouping field '@id' for "Sex" is 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/field_sex'
#
# Replace these with actual @ids from your schema as needed.

main_record_set_id = main_set_id  # e.g., 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/recordset_clinical_data'
# Assign hypothetical field @ids
numeric_field_id = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/field_age_at_second_crc'  # Replace with the actual @id for "Age at Second CRC Diagnosis"
group_field_id = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/field_sex'  # Replace with actual @id for "Sex"

# Check if the field exists
df = dataframes[main_record_set_id].copy()
if numeric_field_id in df.columns:
    # Filtering
    threshold = 60
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
    else:
        print(f"Grouping field {group_field_id} not found in this DataFrame.")
else:
    print(f"Numeric field {numeric_field_id} not found in DataFrame columns:")
    print(df.columns.tolist())

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship with the chosen categorical field, using only their `@id`s for clarity and schema traceability.

In [ ]:
import matplotlib.pyplot as plt

# Check that we have good data for plotting
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=15, alpha=0.7)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group (if grouping field available)
    if group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} vs {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print(f"Numeric field {numeric_field_id} not present for visualization.")

## 6. Conclusion

This notebook has demonstrated loading, schema-aware referencing, and exploratory processing of the FAIR^2 dataset with `mlcroissant`. All elements are referenced by `@id` for complete traceability. Adjust the `@id`s in code cells to match the specific fields of interest in your analysis, as enumerated in the outputs of Section 2.

You can now build on this structure with deeper statistical analysis or integrate ML workflows using these pre-processed DataFrames.